# Sprint 9 · Webinar 27 · Data Analytics teórico · Student Version  
## Validar hipótesis de negocio con pruebas estadísticas

**Programa:** Data Analytics  
**Sprint:** 9  
**Duración sugerida:** 100 minutos  
**Modalidad:** Teórica con demostraciones en Python

> Usa este notebook como cuaderno de apuntes y trabajo guiado.  
> Durante la sesión deberías **anotar conceptos**, **ejecutar demostraciones**, **completar interpretaciones** y **responder preguntas de negocio con evidencia**.


## Fecha

Completa la información de la sesión:

- **Fecha:** `____ / ____ / 2026`
- **Instructor/a:** `____________________________`
- **Grupo / Cohorte:** `____________________________`
- **Tema central:** `Pruebas de hipótesis para negocio`


## Objetivos de la sesión

Al finalizar esta sesión, deberías poder:

1. Convertir una **pregunta de negocio** en una **pregunta medible con datos**.
2. Explicar por qué usamos **pruebas de hipótesis** cuando trabajamos con muestras.
3. Verificar si un dataset está suficientemente listo para soportar una comparación.
4. Diferenciar cuándo conviene usar:
   - **t-test** para promedios,
   - **z-test** para proporciones,
   - **chi-cuadrada** para relaciones entre variables categóricas.
5. Comunicar resultados con lenguaje de negocio, no solo con tecnicismos estadísticos.

### Auto-chequeo inicial
Marca cómo te sientes antes de empezar:

- [ ] Nunca he usado pruebas estadísticas
- [ ] He oído hablar de p-value, pero no lo explico con seguridad
- [ ] Puedo distinguir promedio vs proporción
- [ ] Ya he visto t-test / z-test / chi-cuadrada


## Agenda sugerida (100 minutos)

| Tiempo | Bloque | Contenido | Lo entendí / dudas |
|---:|---|---|---|
| 0–10 | Actividad 0 | De pregunta de negocio a pregunta medible | `__` |
| 10–25 | Dataset | Crear dataset extenso + exploración rápida | `__` |
| 25–45 | Fundamentos | Muestra, población, variabilidad, H0/H1, p-value | `__` |
| 45–70 | Comparar grupos | t-test (promedios) y z-test (proporciones) | `__` |
| 70–85 | Variables categóricas | Chi-cuadrada + supuestos | `__` |
| 85–95 | Comunicación | Cómo reportar resultados a negocio | `__` |
| 95–100 | Cierre | Takeaways y próximos pasos | `__` |


## Actividad 0 · Calentamiento (10 min)

**Objetivo:** activar intuición de medición antes de entrar en estadística formal.

### En equipos o de forma individual
Piensa en un negocio digital y responde:

1. **¿Qué cambio o experimento te gustaría evaluar?**  
   `____________________________________________________________`

2. **¿Qué métrica usarías para medir el efecto?**  
   `____________________________________________________________`

3. **¿Qué grupos compararías?**  
   `____________________________________________________________`

4. **¿Qué resultado te haría pensar que el cambio “funcionó”?**  
   `____________________________________________________________`

### Mini tabla de trabajo

| Caso | Cambio / tratamiento | Métrica | Grupo A | Grupo B | ¿Qué esperas que ocurra? |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |

> Guarda estas respuestas. Más adelante las vamos a traducir a H0 y H1.


In [ ]:

# 1) Setup del entorno
# Ejecuta esta celda primero.
# Si aparece un error de librerías, anota aquí cuál fue y cómo lo resolviste.

# ============================================================
# Imports y configuración
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

RANDOM_SEED = 42

# Nota de clase:
# ¿Qué librería crees que usaremos para cada tipo de tarea?
# - pandas  -> _____________________________________________
# - scipy   -> _____________________________________________
# - matplotlib -> __________________________________________


## Ejercicio 1 · Crear un dataset extenso para practicar (15 min)

**Caso:** experimento A/B de checkout en e-commerce

- Variante **A**: checkout actual
- Variante **B**: checkout nuevo

### Variables del caso
- **Binaria (proporción):** `converted`
- **Numérica (promedio):** `order_value_usd` (solo si hay compra)
- **Categóricas:** `device`, `region`, `traffic_source`
- **Numéricas auxiliares:** `time_on_site_min`, `pages_viewed`

### Tu objetivo en esta sección
1. Ejecutar la generación del dataset.
2. Explorar su estructura.
3. Identificar qué columnas podrían servir para:
   - una prueba de promedios,
   - una prueba de proporciones,
   - una prueba entre variables categóricas.

### Antes de ejecutar
Escribe una predicción rápida:

- La variante B debería mejorar principalmente: `__________________________`
- La variable más útil para una prueba t sería: `__________________________`
- La variable más útil para una chi-cuadrada sería: `_______________________`


In [ ]:
# 2) Generación del dataset sintético
# Lee la función y ubica:
# - qué variable representa el tratamiento (A/B)
# - qué variable representa la conversión
# - qué columnas podrían introducir segmentación
# - dónde aparecen missing values u outliers
#
# Nota de apuntes:
# ¿Qué ventaja tiene usar un dataset sintético para aprender este tema?
# _________________________________________________________________

# ============================================================
# Generación del dataset sintético (controlado y reproducible)
# ============================================================

def generate_sprint9_dataset(n_rows: int = 12000, seed: int = 42) -> pd.DataFrame:
    """
    Genera un dataset sintético para practicar pruebas estadísticas básicas en un contexto de negocio.

    Diseñado para principiantes:
    - Tipos de variables variados: numéricas, binarias, categóricas y ordinales.
    - Un efecto realista: la variante B mejora un poco la conversión (para que el z-test "tenga algo que encontrar").
    - Incluye problemas comunes: missing values y outliers.

    Parámetros:
    - n_rows: número de filas (usuarios)
    - seed: semilla para reproducibilidad
    """

    rng = np.random.default_rng(seed)

    # --- Categóricas ---
    variant = rng.choice(["A", "B"], size=n_rows, p=[0.50, 0.50])
    region = rng.choice(["Andina", "Caribe", "Pacífica", "Orinoquía-Amazonía"], size=n_rows, p=[0.50, 0.18, 0.22, 0.10])
    device = rng.choice(["Mobile", "Desktop", "Tablet"], size=n_rows, p=[0.68, 0.28, 0.04])
    traffic_source = rng.choice(["Organic", "Paid", "Referral", "Email"], size=n_rows, p=[0.42, 0.36, 0.12, 0.10])
    payment_method = rng.choice(["Card", "PSE", "Cash", "Wallet"], size=n_rows, p=[0.60, 0.20, 0.10, 0.10])

    # --- Comportamiento ---
    days_since_signup = rng.integers(0, 365, size=n_rows)
    pages_viewed = np.clip(rng.poisson(lam=6, size=n_rows), 1, 60)

    base_time = 1.2 + 0.35 * pages_viewed + rng.normal(0, 2.0, size=n_rows)
    device_time_adj = np.where(device == "Mobile", -1.0, np.where(device == "Tablet", 0.5, 0.0))
    time_on_site_min = np.clip(base_time + device_time_adj, 0.2, None)

    # --- Conversión (binaria) ---
    # Probabilidad base
    p = np.full(n_rows, 0.085)

    # Ajustes simples por fuente y dispositivo
    p += np.where(traffic_source == "Email", 0.030, 0.0)
    p += np.where(traffic_source == "Paid", 0.010, 0.0)
    p += np.where(device == "Desktop", 0.015, 0.0)
    p += np.where(device == "Mobile", -0.005, 0.0)

    # Efecto de negocio a probar: B mejora conversión
    p += np.where(variant == "B", 0.012, 0.0)

    p = np.clip(p, 0.001, 0.90)
    converted = rng.binomial(n=1, p=p, size=n_rows)

    # --- Valor de orden (solo si convirtió) ---
    order_value = rng.lognormal(mean=3.45, sigma=0.55, size=n_rows)
    order_value *= np.where(payment_method == "Card", 1.05, 1.0)
    order_value *= np.where(region == "Andina", 1.03, 1.0)

    order_value_usd = np.where(converted == 1, order_value, np.nan)

    # --- Tickets de soporte y satisfacción (solo si convirtió) ---
    support_tickets_30d = np.clip(rng.poisson(lam=0.35, size=n_rows), 0, 10)

    satisfaction_latent = (
        3.6
        + np.where(variant == "B", 0.10, 0.0)
        - 0.35 * support_tickets_30d
        + rng.normal(0, 0.75, size=n_rows)
    )

    satisfaction_1_5 = np.clip(np.round(satisfaction_latent), 1, 5).astype(int)
    satisfaction_1_5 = np.where(converted == 1, satisfaction_1_5, np.nan)

    # --- Churn (binaria) ---
    churn_p = np.full(n_rows, 0.14)
    churn_p += np.where(converted == 0, 0.10, 0.0)
    churn_p += np.where(device == "Mobile", 0.02, 0.0)
    churn_p -= np.where((converted == 1) & (satisfaction_1_5 >= 4), 0.06, 0.0)

    churn_p = np.clip(churn_p, 0.01, 0.95)
    churned_30d = rng.binomial(n=1, p=churn_p, size=n_rows)

    df = pd.DataFrame({
        "user_id": np.arange(1, n_rows + 1),
        "variant": variant,
        "region": region,
        "device": device,
        "traffic_source": traffic_source,
        "payment_method": payment_method,
        "days_since_signup": days_since_signup,
        "pages_viewed": pages_viewed,
        "time_on_site_min": np.round(time_on_site_min, 2),
        "converted": converted,
        "order_value_usd": np.round(order_value_usd, 2),
        "support_tickets_30d": support_tickets_30d,
        "satisfaction_1_5": satisfaction_1_5,
        "churned_30d": churned_30d,
    })

    # --- Problemas controlados ---
    # Missing values
    miss_idx_time = rng.choice(df.index, size=int(0.01 * n_rows), replace=False)
    df.loc[miss_idx_time, "time_on_site_min"] = np.nan

    miss_idx_region = rng.choice(df.index, size=int(0.01 * n_rows), replace=False)
    df.loc[miss_idx_region, "region"] = np.nan

    # Outliers en order_value_usd (solo convertidos)
    conv_idx = df.index[df["converted"] == 1]
    if len(conv_idx) > 0:
        out_idx = rng.choice(conv_idx, size=max(1, int(0.003 * len(conv_idx))), replace=False)
        df.loc[out_idx, "order_value_usd"] *= 15

    return df

df = generate_sprint9_dataset(n_rows=12000, seed=RANDOM_SEED)
df.head()


In [ ]:

# 3) Crear el DataFrame
# Ejecuta la generación y registra tus primeras observaciones.

df = generate_sprint9_dataset(n_rows=12000, seed=RANDOM_SEED)

print("Dimensiones:", df.shape)

# Mis observaciones iniciales:
# 1) ____________________________________________
# 2) ____________________________________________
# 3) ____________________________________________


In [ ]:

# 4) Muestra aleatoria del dataset
# Observa algunas filas y anota qué columnas entiendes con claridad
# y cuáles te gustaría preguntarle al docente.

display(df.sample(5, random_state=RANDOM_SEED))

# Columnas claras: ______________________________________________
# Columnas que debo aclarar: ____________________________________


In [ ]:

# 5) Tipos de variables
# Después de ejecutar, clasifica al menos 4 columnas:
# - binaria
# - numérica continua
# - categórica nominal
# - identificador

print("\nTipos de variables:")
display(df.dtypes)

# Mis ejemplos:
# - Binaria: __________________________________
# - Numérica: _________________________________
# - Categórica: _______________________________
# - ID / llave: _______________________________


In [ ]:

# 6) Información general del DataFrame
# Usa esta salida para responder:
# - ¿cuántas columnas no nulas hay?
# - ¿ves alguna columna con muchos faltantes?
# - ¿algún tipo de dato debería revisarse?

df.info()

# Resumen en mis palabras:
# _______________________________________________________________


In [ ]:

# 7) Missing values por columna
# Anota cuáles columnas tienen faltantes y si esos faltantes
# parecen lógicos o problemáticos para el análisis.

missing_pct = (df.isna().mean().sort_values(ascending=False) * 100).to_frame("missing_%").round(2)
print("\nPorcentaje de missing values por columna:")
display(missing_pct)

# Columnas con faltantes esperados: ______________________________
# Columnas con faltantes que me preocuparían: ____________________


## 9.1 Formular preguntas de negocio como hipótesis

Antes de hablar de fórmulas o pruebas, necesitamos traducir una necesidad del negocio a algo que podamos medir con datos.

### Idea base
1. El negocio hace un cambio o compara grupos.
2. Elegimos una métrica.
3. Observamos una diferencia.
4. Evaluamos si esa diferencia parece real o podría explicarse por azar.

### Completa con tus palabras

| Concepto | Mi explicación |
|---|---|
| Pregunta de negocio | |
| Métrica | |
| Comparación | |
| Diferencia observada | |
| Variabilidad / azar | |

### Ejemplo rápido
> “¿El checkout B mejora la conversión frente a A?”

- Métrica: `____________________________________________`
- Grupo 1: `____________________________________________`
- Grupo 2: `____________________________________________`
- Tipo de variable: `_____________________________________`


### 9.1.1 ¿Por qué necesitamos pruebas de hipótesis? (Muestra vs. población)

Completa esta tabla durante la explicación:

| Término | Definición en tus palabras | Ejemplo en este caso |
|---|---|---|
| Población | | |
| Muestra | | |
| Variabilidad muestral | | |
| Incertidumbre | | |

### Pregunta guía
¿Por qué no basta con ver que “B tiene un número más alto” para concluir que realmente ganó?

`_______________________________________________________________`

`_______________________________________________________________`


### 9.1.2 Introducción al testing de hipótesis

Cuando formalizamos una comparación, solemos definir:

- **H0 (hipótesis nula):** no hay diferencia / no hay efecto.
- **H1 (hipótesis alternativa):** sí hay diferencia / sí hay efecto.

Y además aparecen estos conceptos:

| Concepto | Definición breve | Mi ejemplo |
|---|---|---|
| α (alfa) | nivel de significancia | |
| p-value | evidencia contra H0 | |
| Error Tipo I | falso positivo | |
| Error Tipo II | falso negativo | |

### Nota de comprensión
Explica con una frase simple qué significa:

- **“rechazar H0”**: `_________________________________________`
- **“no rechazar H0”**: `______________________________________`


In [ ]:

# Ejemplo guiado: traducir una pregunta de negocio a H0/H1
# Lee el ejemplo y luego escribe uno propio debajo.

business_question = "¿El checkout B aumenta la tasa de conversión frente a A?"
H0 = "La tasa de conversión es igual en A y B (p_A = p_B)."
H1 = "La tasa de conversión es mayor en B que en A (p_B > p_A)."

print("Pregunta de negocio:", business_question)
print("H0:", H0)
print("H1:", H1)

# Ahora escribe tu propio ejemplo:
# Pregunta: _________________________________________________
# H0: _______________________________________________________
# H1: _______________________________________________________


### 9.1.3 Validar si los datos están listos para probar una hipótesis

Antes de correr cualquier prueba, revisa este checklist:

- [ ] La métrica está bien definida.
- [ ] Los grupos están claros.
- [ ] No hay problemas graves de calidad de datos.
- [ ] La unidad de análisis tiene sentido (ej. 1 fila = 1 usuario).
- [ ] Los supuestos básicos de la prueba son razonables.

### Completa esta tabla

| Revisión | ¿Qué debo mirar? | ¿Por qué importa? |
|---|---|---|
| Definición de la métrica | | |
| Missing values | | |
| Duplicados | | |
| Outliers | | |
| Diseño del experimento | | |
| Supuestos estadísticos | | |


In [ ]:

# 8) Checklist de data readiness: duplicados
# Una fila repetida o un user_id duplicado puede sesgar la comparación.

dup_rate = df["user_id"].duplicated().mean()
print(f"Tasa de user_id duplicados: {dup_rate:.4f}")

# Interpretación:
# ¿Este porcentaje te preocuparía? ¿Por qué?
# _____________________________________________________________


In [ ]:

# 9) Missing values: conteo por columna
# Observa el volumen y anota si afecta o no el análisis.

missing_counts = df.isna().sum().sort_values(ascending=False)
display(missing_counts.to_frame("missing_count").head(10))

# Mi lectura:
# _____________________________________________________________


In [ ]:

# 10) Balance del experimento
# Si A/B estuvo razonablemente balanceado, esperamos tamaños y promedios
# parecidos en variables de contexto.

balance_n = df.groupby("variant")["user_id"].count()
balance_pages = df.groupby("variant")["pages_viewed"].mean()
balance_time = df.groupby("variant")["time_on_site_min"].mean()
mobile_rate = (
    df.assign(is_mobile=(df["device"] == "Mobile").astype(int))
      .groupby("variant")["is_mobile"].mean()
)

balance = pd.DataFrame({
    "n_users": balance_n,
    "avg_pages_viewed": balance_pages.round(2),
    "avg_time_on_site_min": balance_time.round(2),
    "mobile_share": mobile_rate.round(4)
})

balance

# ¿El experimento luce balanceado?
# ______________________________________


In [ ]:

# 11) Espacio para notas del balance
# Si quieres, agrega aquí una validación extra propuesta por el docente.
# Ejemplo: comparar una categoría adicional por variante.

balance


In [ ]:

# 12) Visual de order_value_usd en usuarios que convirtieron
# Esto ayuda a discutir asimetría, colas largas y outliers.

orders = df.loc[df["converted"] == 1, "order_value_usd"].dropna()

plt.figure()
plt.hist(orders, bins=60)
plt.title("Distribución de order_value_usd (solo convertidos)")
plt.xlabel("order_value_usd")
plt.ylabel("frecuencia")
plt.show()

print("Resumen order_value_usd (convertidos):")
display(orders.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

# ¿La distribución parece simétrica?
# ______________________________________
# ¿Ves indicios de outliers?
# ______________________________________


### 9.1.4 Cómo elegir la prueba estadística correcta

Usa esta guía rápida:

1. **¿Comparo promedios?**  
   Ejemplo: ticket promedio en A vs B  
   → **t-test**

2. **¿Comparo proporciones?**  
   Ejemplo: conversión en A vs B  
   → **z-test de proporciones**

3. **¿Busco relación entre dos categóricas?**  
   Ejemplo: dispositivo vs convertido sí/no  
   → **chi-cuadrada**

### Completa la tabla

| Pregunta | Tipo de variable | Comparación | Prueba sugerida | ¿Por qué? |
|---|---|---|---|---|
| ¿B cambia el ticket promedio? | | | | |
| ¿B mejora la conversión? | | | | |
| ¿La conversión depende del dispositivo? | | | | |


## 9.2 Comparando promedios y proporciones con Python

En esta sección vas a ver dos patrones muy comunes en analítica:

- comparar **promedios** entre grupos,
- comparar **tasas / proporciones** entre grupos.

En ambos casos no basta con ver una diferencia visual: necesitamos decidir si la evidencia es suficiente.


### 9.2.1 Prueba t para comparar promedios (t-test)

**Pregunta del ejemplo:**  
> Entre quienes compran, ¿el ticket promedio es distinto entre A y B?

### Recuerda
- La variable es **numérica**.
- Solo usamos filas con compra, porque `order_value_usd` solo existe cuando hay conversión.
- La prueba evalúa si la diferencia observada entre medias es compatible con azar.

### Antes de ejecutar
Predice:

- Creo que el promedio en A será mayor / menor / similar a B: `____________`
- Si la diferencia es pequeña, esperaría un p-value: `alto / bajo`


In [ ]:

# 13) Preparar datos para t-test
# Dejamos únicamente la variable numérica y la variante correspondiente.

df_orders = df.loc[df["converted"] == 1, ["variant", "order_value_usd"]].dropna()
df_orders.head()

# ¿Cuántas filas quedaron?
# ________________________


In [ ]:

# 14) Separar muestras A y B
# Estas dos series son las que compararemos.

a = df_orders[df_orders["variant"] == "A"]["order_value_usd"]
b = df_orders[df_orders["variant"] == "B"]["order_value_usd"]

print("Series preparadas.")
print("A ->", a.shape)
print("B ->", b.shape)


In [ ]:

# 15) Tamaño de muestra por grupo
# El tamaño de muestra influye en la estabilidad de la estimación.

print("Tamaño de muestra (convertidos) A:", len(a))
print("Tamaño de muestra (convertidos) B:", len(b))

# ¿Los tamaños son parecidos?
# ____________________________________


In [ ]:

# 16) Medias por grupo
# Observa la diferencia cruda antes de aplicar la prueba.

print("Promedio A:", round(a.mean(), 2))
print("Promedio B:", round(b.mean(), 2))

# Diferencia observada a simple vista:
# ____________________________________


In [ ]:

# 17) Chequeo simple de varianzas: Levene
# Si las varianzas parecen diferentes, suele preferirse Welch (equal_var=False).

stat_levene, p_levene = stats.levene(a, b, center="median")
print(f"Levene p-value (varianzas iguales): {p_levene:.4f}")

# Registra la regla:
# Si p_levene < 0.05, entonces __________________________________


In [ ]:

# 18) Decidir si asumir varianzas iguales
equal_var = p_levene >= 0.05
print("¿Usar equal_var=True?", equal_var)


In [ ]:

# 19) Ejecutar el t-test
# Completa la prueba usando la variable equal_var calculada antes.

t_stat, p_value = stats.ttest_ind(a, b, equal_var=equal_var)
print(f"t-stat: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"equal_var usado: {equal_var}")

# Decisión preliminar:
# ____________________________________


### 9.2.2 Interpretando los resultados de una prueba t

No reportes solo el p-value.

También conviene comunicar:

- **Diferencia de medias** (en unidades del negocio, aquí USD).
- **Intervalo de confianza** para la diferencia.
- Una interpretación clara para el stakeholder.

### Plantilla de interpretación
> “La variante ___ tuvo un ticket promedio ___ USD mayor/menor que ___.
> El p-value fue ___, por lo que ______.
> En términos de negocio, esto sugiere que ______.”


In [ ]:

# 20) Diferencia de medias + IC aproximado al 95%
alpha = 0.05

mean_diff = b.mean() - a.mean()
print(f"Diferencia de medias (B - A): {mean_diff:.2f} USD")

# IC 95% aproximado para diferencia de medias (Welch)
se = np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
df_welch = (a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))**2 / (
    (a.var(ddof=1)/len(a))**2/(len(a)-1) + (b.var(ddof=1)/len(b))**2/(len(b)-1)
)
t_crit = stats.t.ppf(1 - alpha/2, df=df_welch)
ci_low = mean_diff - t_crit * se
ci_high = mean_diff + t_crit * se

print(f"IC 95% (aprox) para B - A: [{ci_low:.2f}, {ci_high:.2f}]")

# Mi lectura:
# - ¿El IC incluye 0? ______________________________
# - ¿Qué implicación tiene eso? _____________________


### 9.2.3 Prueba z para comparar proporciones

**Pregunta del ejemplo:**  
> ¿La tasa de conversión en B es mayor que en A?

### Recuerda
- La variable aquí es **binaria** (`converted`).
- Lo importante es comparar la **proporción de éxito** en cada grupo.
- En este ejemplo usaremos una hipótesis direccional: **B > A**.


In [ ]:

# 21) Vista rápida del dataset antes del z-test
# Solo para recordar la estructura de las columnas implicadas.

df.head()

# ¿Qué columnas necesito para esta prueba?
# ________________________________________


In [ ]:

# 22) Conteos de convertidos y total por variante
conv_stats = df.groupby("variant")["converted"].agg(["sum", "count"])
display(conv_stats)

# Reordenamos explícitamente a [A, B]
conv_stats = conv_stats.loc[["A", "B"]]

success = conv_stats["sum"].values
nobs = conv_stats["count"].values

rate_A = success[0] / nobs[0]
rate_B = success[1] / nobs[1]

print(f"Conversion rate A: {rate_A:.4%}")
print(f"Conversion rate B: {rate_B:.4%}")
print(f"Diferencia absoluta (B - A): {(rate_B - rate_A):.4%}")

# En puntos porcentuales, el cambio es:
# ________________________________________


In [ ]:

# 23) Verifica los arrays que usará proportions_ztest
success


In [ ]:

# 24) Número de observaciones por grupo
nobs


In [ ]:

# 25) Ejecutar z-test de proporciones
# H1: p_B > p_A

z_stat, p_z = proportions_ztest(count=success, nobs=nobs, alternative="larger")
print(f"z-stat: {z_stat:.4f}")
print(f"p-value: {p_z:.6f}")

# Interpretación preliminar:
# ________________________________________


### 9.2.4 Interpretando los resultados de una prueba z

Para comunicar a negocio, evita frases como “el modelo dio significativo” sin contexto.

### Intenta completar esta estructura
- Conversión A: `________________`
- Conversión B: `________________`
- Diferencia en puntos porcentuales: `________________`
- p-value: `________________`
- Conclusión de negocio: `___________________________________________`

> Recuerda: **evidencia estadística** no siempre implica **impacto relevante para negocio**.


In [ ]:

# 26) Decisión formal usando alpha
alpha = 0.05
decision = (
    "RECHAZAR H0 (evidencia de mejora en B)"
    if p_z < alpha
    else "NO rechazar H0 (evidencia insuficiente)"
)

print(f"Decisión con α={alpha}: {decision}")

# ¿Cómo se lo explicarías a un stakeholder no técnico?
# _____________________________________________________


## 9.3 Comprobando relaciones categóricas con Python

Ahora no compararemos medias ni tasas entre dos variantes, sino la relación entre dos variables categóricas.

### Pregunta guía
> ¿La conversión depende del dispositivo?

Aquí la idea es ver si la distribución observado vs esperado cambia lo suficiente como para pensar que hay asociación.


### 9.3.1 Prueba Chi-cuadrada (χ²) para variables categóricas

Pasos:

1. Construir una **tabla de contingencia**.
2. Ejecutar la prueba χ².
3. Revisar el p-value.
4. Verificar supuestos mínimos.
5. Traducir el hallazgo a una conclusión entendible.

### Antes de ejecutar
¿Qué esperarías encontrar en este caso?

`_______________________________________________________________`


In [ ]:

# 27) Tabla de contingencia + prueba chi-cuadrada
ct = pd.crosstab(df["device"], df["converted"])
display(ct)

chi2, p_chi, dof, expected = stats.chi2_contingency(ct)
print(f"chi2: {chi2:.4f}")
print(f"dof: {dof}")
print(f"p-value: {p_chi:.6f}")

expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)
display(expected_df.round(2))

# ¿Los conteos observados y esperados parecen muy distintos?
# __________________________________________________________


### 9.3.2 Supuestos y limitaciones de χ²

Marca lo importante:

- [ ] Las observaciones deberían ser independientes.
- [ ] Las categorías no deberían tener conteos extremadamente bajos.
- [ ] Los conteos esperados idealmente deberían ser razonables.

### Completa
Si varias celdas tienen conteos esperados muy pequeños, una posible acción es:

`_______________________________________________________________`


In [ ]:

# 28) Revisar celdas esperadas pequeñas + ejemplo de agrupación
expected_small = (expected < 5).sum()
total_cells = expected.size
print(f"Celdas con conteo esperado < 5: {expected_small}/{total_cells}")

# Ejemplo de agrupación simple: Tablet -> Desktop
df2 = df.copy()
df2["device_grouped"] = df2["device"].replace({"Tablet": "Desktop"})

ct2 = pd.crosstab(df2["device_grouped"], df2["converted"])
chi2_2, p_chi_2, dof_2, expected_2 = stats.chi2_contingency(ct2)

print("\nTabla agrupada (Tablet -> Desktop):")
display(ct2)

print(f"chi2 agrupado: {chi2_2:.4f}")
print(f"p-value agrupado: {p_chi_2:.6f}")

# ¿Cambió algo relevante al agrupar?
# ____________________________________


### 9.3.3 Visualizar relaciones categóricas para comunicar negocio

A un stakeholder suele servirle más ver **tasas** que ver solo conteos.

### Pregunta guía
¿Por qué una tasa por dispositivo suele ser más útil que una tabla bruta de conteos?

`_______________________________________________________________`


In [ ]:

# 29) Tasa de conversión por dispositivo
conv_rate_by_device = df.groupby("device")["converted"].mean().sort_values(ascending=False)

plt.figure()
plt.bar(conv_rate_by_device.index, conv_rate_by_device.values)
plt.title("Tasa de conversión por dispositivo")
plt.xlabel("Dispositivo")
plt.ylabel("Conversion rate")
plt.ylim(0, conv_rate_by_device.max() * 1.25)
plt.show()

display(conv_rate_by_device.to_frame("conversion_rate"))

# ¿Qué dispositivo muestra mejor tasa?
# ____________________________________
# ¿Eso prueba causalidad? ¿Por qué no?
# ____________________________________


## Takeaways

Completa al final de la sesión:

1. Una prueba de hipótesis sirve para  
   `____________________________________________________________`

2. La diferencia entre muestra y población importa porque  
   `____________________________________________________________`

3. Cuando comparo un promedio, normalmente pienso en  
   `____________________________________________________________`

4. Cuando comparo una proporción, normalmente pienso en  
   `____________________________________________________________`

5. Cuando comparo dos categóricas, normalmente pienso en  
   `____________________________________________________________`

6. Un p-value por sí solo no basta porque  
   `____________________________________________________________`


## Cierre y próximos pasos

### Antes de la próxima sesión
Marca lo que te conviene repasar:

- [ ] H0 vs H1
- [ ] p-value y nivel α
- [ ] Diferencia entre promedio y proporción
- [ ] t-test
- [ ] z-test
- [ ] chi-cuadrada
- [ ] Interpretación de resultados para negocio

### Mini reflexión
1. La parte más clara de la clase fue:  
   `____________________________________________________________`

2. La parte que aún necesito practicar es:  
   `____________________________________________________________`

3. Una pregunta de negocio que ahora sí podría probar con datos es:  
   `____________________________________________________________`

### ¿Necesitas ayuda?
- `DATA-CO-LEARNING`: revisa los horarios de apoyo en la guía del estudiante.
- `DA_CONSULTA`: publica dudas de contenido o del proyecto usando el tag correspondiente.
- `SPRINT FOCUS`: sesiones extra para profundizar temas clave del sprint.
- `SESIONES 1:1`: agenda con anticipación si necesitas apoyo individual.


## Siguientes pasos

- **Próxima clase sugerida:** aplicar estas pruebas en un notebook práctico.
- **Recomendación:** vuelve a ejecutar este notebook y cambia la pregunta de negocio.
- **Reto personal:** inventa un caso nuevo y decide qué prueba usarías antes de programarla.

### Espacio final de apuntes
`______________________________________________________________`

`______________________________________________________________`

`______________________________________________________________`
